# Trabajo Práctico Integrador – Entrega 5

**Materia:** Introducción al Análisis de Datos  
**Tema:** Cancelación de reservas hoteleras  
**Comisión:** 10 (Grupo J)  
**Integrante:** Viruel, Nicolás  
**Entrega:** 5 – Primer modelo predictivo e interpretación inicial  
**Recorrido:** SQL (DuckDB) + Python  
**Dataset:** `hotel_booking_TPI_grupo_J.csv`  
**Variable objetivo:** `is_canceled`  
**Año:** 2026

> Notebook **independiente y reproducible** desde cero. Sigue el patrón de la práctica de Semana 12: seleccionar datos → preparar X e y → dividir entrenamiento/prueba → entrenar → predecir → interpretar. **No** se evalúa el modelo con métricas formales (Unidad N° 6).

## 1. Objetivo de esta entrega

El hotel necesita **anticipar**, al momento de registrar una reserva, si esa reserva será cancelada antes del check-in. El problema es de **clasificación supervisada**: la etiqueta `is_canceled` ya está en el dataset (`1` = cancelada, `0` = no cancelada).

En esta entrega se construye un **primer modelo sencillo** (árbol de decisión de profundidad limitada), se observan predicciones frente al resultado real y se interpretan utilidades y limitaciones. El foco no es obtener el mejor modelo posible.

## 2. Entorno, carga del dataset y registro en DuckDB

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

for package in ['duckdb', 'pandas', 'scikit-learn']:
    module = 'sklearn' if package == 'scikit-learn' else package
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

import duckdb
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

CANDIDATES = [
    Path('../datos') / 'hotel_booking_TPI_grupo_J.csv',
    Path('datos') / 'hotel_booking_TPI_grupo_J.csv',
    Path('hotel_booking_TPI_grupo_J.csv'),
]
CSV_PATH = next((p for p in CANDIDATES if p.exists()), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        'No se encontró hotel_booking_TPI_grupo_J.csv. '
        'Colocá el CSV en ../datos, ./datos o junto al notebook.'
    )

con = duckdb.connect(database=':memory:')
con.execute(
    '''
    CREATE OR REPLACE TABLE reservas AS
    SELECT * FROM read_csv_auto(?, header=true)
    ''',
    [str(CSV_PATH.resolve())],
)
n_filas = con.execute('SELECT COUNT(*) FROM reservas').fetchone()[0]
n_cols = con.execute('SELECT COUNT(*) FROM (DESCRIBE reservas)').fetchone()[0]
print(f'CSV: {CSV_PATH.resolve()}')
print(f'Filas cargadas en DuckDB: {n_filas:,} · columnas: {n_cols}')

CSV: /run/media/nicolas/Datos/moodle-utn-mcp/materias/iad/semana-12/datos/hotel_booking_TPI_grupo_J.csv
Filas cargadas en DuckDB: 25,000 · columnas: 32


## 3. Paso 1 · Preparar los datos para el modelo

### 3.1 Variable objetivo y predictores

La unidad de análisis es **una reserva hotelera** (una fila). La etiqueta `y` es `is_canceled`.

Se usan **siete predictores** alineados con la planificación de Semana 11 y con los hallazgos del EDA (Entrega 4). `total_nights` se deriva en SQL como `stays_in_weekend_nights + stays_in_week_nights`.

| Predictor | Por qué entra |
|-----------|----------------|
| `lead_time` | En el EDA, la mediana de anticipación fue más alta en las canceladas. |
| `hotel` | Tasas distintas entre City Hotel y Resort Hotel. |
| `deposit_type` | Se pacta al reservar; en este subset las tasas difieren fuerte (`Non Refund` vs `No Deposit`). |
| `market_segment` | Segmentos con tasas y volumen suficientes. |
| `total_nights` | Duración pedida, conocida al registrar; candidata en el EDA. |
| `previous_cancellations` | Historial previo, si el hotel ya lo tiene. |
| `customer_type` | Tipo de cliente identificado al tomar la reserva. |

Criterio de control: **¿el hotel ya conocería este dato cuando recibe la reserva?** Si el valor se genera o se actualiza después, no entra.

### 3.2 Variables que no se usan (fuga o falta de señal)

- **`booking_id`:** identificador de la reserva. No describe un patrón repetible.
- **`assigned_room_type`:** puede asignarse o cambiarse **después** del registro. Al alta el dato disponible es `reserved_room_type`. Usar la habitación finalmente asignada sería **fuga de información** (*data leakage*).
- **`booking_changes`:** cuenta cambios posteriores. Ese valor no está cerrado cuando acaba de registrarse la reserva.
- **`is_canceled`:** es la respuesta que se quiere anticipar; no puede ser predictora.
- **`agent` / `company`:** muchos nulos en el subset; no se usan en este primer modelo.

In [2]:
df = con.sql('''
SELECT
    is_canceled AS objetivo,
    lead_time,
    hotel,
    deposit_type,
    market_segment,
    stays_in_weekend_nights + stays_in_week_nights AS total_nights,
    previous_cancellations,
    customer_type
FROM reservas
''').df()

con.register('modelo_df', df)
print('Filas para el modelo:', len(df))
print('Columnas:', list(df.columns))
df.head()

Filas para el modelo: 25000
Columnas: ['objetivo', 'lead_time', 'hotel', 'deposit_type', 'market_segment', 'total_nights', 'previous_cancellations', 'customer_type']


,objetivo,lead_time,hotel,deposit_type,market_segment,total_nights,previous_cancellations,customer_type
0,0,261,Resort Hotel,No Deposit,Offline TA/TO,7,0,Contract
1,1,85,Resort Hotel,No Deposit,Groups,3,0,Transient-Party
2,1,364,City Hotel,Non Refund,Offline TA/TO,2,1,Transient-Party
3,0,32,City Hotel,No Deposit,Online TA,2,0,Transient
4,1,126,City Hotel,No Deposit,Offline TA/TO,2,0,Transient


### 3.3 Valores faltantes y variables categóricas

Antes de entrenar hay que revisar nulos y el tipo de cada predictor. Las categóricas (`hotel`, `deposit_type`, `market_segment`, `customer_type`) se convertirán a dummies **después** de partir train/test, usando solo las categorías del entrenamiento.

In [3]:
nulos = con.sql('''
SELECT
    COUNT(*) AS filas,
    SUM(CASE WHEN objetivo IS NULL THEN 1 ELSE 0 END) AS nulos_objetivo,
    SUM(CASE WHEN lead_time IS NULL THEN 1 ELSE 0 END) AS nulos_lead_time,
    SUM(CASE WHEN hotel IS NULL THEN 1 ELSE 0 END) AS nulos_hotel,
    SUM(CASE WHEN deposit_type IS NULL THEN 1 ELSE 0 END) AS nulos_deposit,
    SUM(CASE WHEN market_segment IS NULL THEN 1 ELSE 0 END) AS nulos_segment,
    SUM(CASE WHEN total_nights IS NULL THEN 1 ELSE 0 END) AS nulos_nights,
    SUM(CASE WHEN previous_cancellations IS NULL THEN 1 ELSE 0 END) AS nulos_prev_cancel,
    SUM(CASE WHEN customer_type IS NULL THEN 1 ELSE 0 END) AS nulos_customer
FROM modelo_df
''').df()
display(nulos)

print('\nTipos:')
print(df.dtypes)
print('\nBalance de la etiqueta:')
print(df['objetivo'].value_counts().sort_index())

,filas,nulos_objetivo,nulos_lead_time,nulos_hotel,nulos_deposit,nulos_segment,nulos_nights,nulos_prev_cancel,nulos_customer
0,25000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



Tipos:
objetivo                  int64
lead_time                 int64
hotel                       str
deposit_type                str
market_segment              str
total_nights              int64
previous_cancellations    int64
customer_type               str
dtype: object

Balance de la etiqueta:
objetivo
0    15741
1     9259
Name: count, dtype: int64


## 4. Paso 2 · Partición entrenamiento / prueba

- `test_size = 0.30` y `random_state = 42` para que la partición sea **reproducible**.
- `stratify=y` conserva la proporción de canceladas en ambos conjuntos.
- Si hubiera nulos numéricos, se completarían con la **mediana del entrenamiento** y se aplicaría el mismo valor al conjunto de prueba. Usar la mediana del dataset completo sería fuga de información.
- Las dummies se arman con las categorías de **train**; el test se reindexa a esas columnas (`fill_value=0`) para alinear el esquema.

In [4]:
y = df['objetivo'].astype(int)
X = df.drop(columns='objetivo')

CATS = ['hotel', 'deposit_type', 'market_segment', 'customer_type']
NUMS = ['lead_time', 'total_nights', 'previous_cancellations']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

X_train = X_train.copy()
X_test = X_test.copy()

# Imputación numérica solo con estadísticas de entrenamiento (aunque en este
# subset los predictores elegidos no tienen nulos: queda el procedimiento).
medianas_train = X_train[NUMS].median()
X_train[NUMS] = X_train[NUMS].fillna(medianas_train)
X_test[NUMS] = X_test[NUMS].fillna(medianas_train)

X_train = pd.get_dummies(X_train, columns=CATS, drop_first=False)
X_test = pd.get_dummies(X_test, columns=CATS, drop_first=False)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print('X_train:', X_train.shape, '| y_train:', y_train.shape)
print('X_test :', X_test.shape, '| y_test :', y_test.shape)
print('Canceladas en train:', int(y_train.sum()), f'({y_train.mean():.1%})')
print('Canceladas en test :', int(y_test.sum()), f'({y_test.mean():.1%})')
print('Columnas después de dummies:')
print(list(X_train.columns))

X_train: (17500, 19) | y_train: (17500,)
X_test : (7500, 19) | y_test : (7500,)
Canceladas en train: 6481 (37.0%)
Canceladas en test : 2778 (37.0%)
Columnas después de dummies:
['lead_time', 'total_nights', 'previous_cancellations', 'hotel_City Hotel', 'hotel_Resort Hotel', 'deposit_type_No Deposit', 'deposit_type_Non Refund', 'deposit_type_Refundable', 'market_segment_Aviation', 'market_segment_Complementary', 'market_segment_Corporate', 'market_segment_Direct', 'market_segment_Groups', 'market_segment_Offline TA/TO', 'market_segment_Online TA', 'customer_type_Contract', 'customer_type_Group', 'customer_type_Transient', 'customer_type_Transient-Party']


## 5. Paso 3 · Entrenar un árbol de decisión sencillo

Se usa `DecisionTreeClassifier` con `max_depth=3` y `random_state=42`, igual que en la práctica de Semana 12. La profundidad limitada facilita leer las reglas; no busca el máximo ajuste.

In [5]:
modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X_train, y_train)

predicciones = modelo.predict(X_test)
probabilidad = modelo.predict_proba(X_test)[:, 1]

print('Modelo entrenado.')
print('\nReglas del árbol (profundidad 3):')
print(export_text(modelo, feature_names=list(X_train.columns), decimals=2))

Modelo entrenado.

Reglas del árbol (profundidad 3):
|--- deposit_type_Non Refund <= 0.50
|   |--- lead_time <= 11.50
|   |   |--- lead_time <= 7.50
|   |   |   |--- class: 0
|   |   |--- lead_time >  7.50
|   |   |   |--- class: 0
|   |--- lead_time >  11.50
|   |   |--- market_segment_Online TA <= 0.50
|   |   |   |--- class: 0
|   |   |--- market_segment_Online TA >  0.50
|   |   |   |--- class: 0
|--- deposit_type_Non Refund >  0.50
|   |--- customer_type_Transient-Party <= 0.50
|   |   |--- class: 1
|   |--- customer_type_Transient-Party >  0.50
|   |   |--- hotel_Resort Hotel <= 0.50
|   |   |   |--- class: 1
|   |   |--- hotel_Resort Hotel >  0.50
|   |   |   |--- class: 0



## 6. Paso 4 · Observar predicciones frente al resultado real

Se comparan **12 casos de prueba**. La probabilidad estimada no es una verdad ni una orden de acción: indica cuánto se inclina el modelo por la clase 1. Valores cercanos a 0,5 marcan predicciones menos definidas.

Todavía **no** se reportan accuracy, precision, recall ni F1.

In [6]:
resultados = X_test.copy()
resultados['resultado_real'] = y_test.to_numpy()
resultados['prediccion'] = predicciones
resultados['probabilidad_clase_1'] = probabilidad.round(3)
resultados['coincide'] = resultados['resultado_real'] == resultados['prediccion']
resultados['distancia_a_0_5'] = (resultados['probabilidad_clase_1'] - 0.5).abs()

con.register('predicciones', resultados.reset_index(drop=True))

muestra = con.sql('''
SELECT
    resultado_real,
    prediccion,
    probabilidad_clase_1,
    coincide,
    CASE
        WHEN probabilidad_clase_1 < 0.30 THEN 'Probabilidad baja'
        WHEN probabilidad_clase_1 > 0.70 THEN 'Probabilidad alta'
        ELSE 'Caso incierto'
    END AS lectura_inicial
FROM predicciones
ORDER BY distancia_a_0_5 ASC
LIMIT 12
''').df()
display(muestra)

,resultado_real,prediccion,probabilidad_clase_1,coincide,lectura_inicial
0,0,0,0.438,True,Caso incierto
1,1,0,0.438,False,Caso incierto
2,1,0,0.438,False,Caso incierto
3,0,0,0.438,True,Caso incierto
4,1,0,0.438,False,Caso incierto
5,1,0,0.438,False,Caso incierto
6,0,0,0.438,True,Caso incierto
7,0,0,0.438,True,Caso incierto
8,1,0,0.438,False,Caso incierto
9,1,0,0.438,False,Caso incierto


In [7]:
extremos = con.sql('''
(SELECT resultado_real, prediccion, probabilidad_clase_1, coincide
 FROM predicciones ORDER BY probabilidad_clase_1 ASC LIMIT 4)
UNION ALL
(SELECT resultado_real, prediccion, probabilidad_clase_1, coincide
 FROM predicciones ORDER BY probabilidad_clase_1 DESC LIMIT 4)
''').df()
print('Casos con probabilidad más baja y más alta:')
display(extremos)

desacuerdos = con.sql('''
SELECT resultado_real, prediccion, COUNT(*) AS cantidad
FROM predicciones
GROUP BY resultado_real, prediccion
ORDER BY resultado_real, prediccion
''').df()
print('\nConteo descriptivo real vs predicción (sin métricas de evaluación):')
display(desacuerdos)

Casos con probabilidad más baja y más alta:


,resultado_real,prediccion,probabilidad_clase_1,coincide
0,0,0,0.082,True
1,0,0,0.082,True
2,0,0,0.082,True
3,0,0,0.082,True
4,1,1,1.000,True
5,1,1,1.000,True
6,1,1,1.000,True
7,1,1,1.000,True



Conteo descriptivo real vs predicción (sin métricas de evaluación):


,resultado_real,prediccion,cantidad
0,0,0,4720
1,0,1,2
2,1,0,1845
3,1,1,933


## 7. Paso 5 · Interpretación, utilidad y limitaciones

### 7.1 Qué representa `y` y qué hace el modelo

`y` (`objetivo`) vale **1** si la reserva fue cancelada y **0** si se concretó. El árbol asigna a cada reserva de prueba una **categoría predicha** y una **probabilidad** de clase 1. La categoría es un recorte de esa probabilidad (umbral 0,5 por defecto); la probabilidad muestra cuán definido está el recorte.

Las variables categóricas se transformaron a dummies porque el árbol de `scikit-learn` trabaja con entradas numéricas. Sin esa conversión el entrenamiento no sería válido con este estimador.

### 7.2 Lectura de los casos observados

En la muestra ordenada por cercanía a 0,5 aparecen reservas **inciertas**: el modelo no se inclina con fuerza. Actuar ahí (por ejemplo, ofrecer un beneficio de retención o bloquear un overbooking) exigiría más contexto: costo de equivocarse, política comercial y datos que el modelo no ve (motivo de viaje, cambios de tarifa, contacto posterior).

La primera regla del árbol separa `deposit_type_Non Refund`. En este subset esa categoría tiene una tasa de cancelación cercana al 100 %, así que el modelo predice clase 1 ahí. Es una **asociación muy fuerte**, no una prueba de que el depósito no reembolsable **cause** la cancelación: puede mezclarse con canal, tipo de cliente o con la forma en que se registran esas reservas. Actuar forzando depósitos no reembolsables a partir de esta rama sería un salto causal injustificado.

Fuera de `Non Refund`, el árbol de profundidad 3 tiende a predecir clase 0 (no cancelada), coherente con que esa es la mayoría del dataset (~63 %). Por eso aparecen desacuerdos: reservas `No Deposit` que sí se cancelaron quedan clasificadas como 0. El desacuerdo no indica que el procedimiento esté mal ejecutado; muestra que un árbol pequeño deja casos fuera.

### 7.3 Utilidad inicial para el hotel

El modelo puede servir como **señal temprana** al registrar la reserva: priorizar seguimiento en casos de alta probabilidad y no tratar igual a todas las reservas. Todavía **no** justifica una política automática (cobrar depósitos, rechazar segmentos, overbooking) sin revisar costos, capacidad operativa y sesgos.

### 7.4 Limitaciones (al menos dos)

1. **Información que el dataset no registra.** El CSV no incluye, entre otras cosas, el motivo de la estadía, quejas, cambios de precio posteriores, competencia local ni el contacto del huésped después de reservar. Esas omisiones limitan tanto la predicción como cualquier acción.
2. **Contexto histórico y cambio de política.** El modelo se entrenó con reservas de este subset (comisión 10, Grupo J). Si el hotel cambia tarifas, depósitos o canales, o si entra un período distinto, las reglas pueden dejar de servir. Los datos históricos no garantizan el mismo patrón a futuro.
3. **Errores con costos distintos.** Predecir cancelación cuando la reserva se iba a concretar (falsa alarma) no cuesta lo mismo que no detectar una cancelación real (pérdida de ocupación o de ingreso). Esta entrega no elige un umbral ni evalúa esos costos.
4. **Árbol pequeño y sin evaluación formal.** `max_depth=3` facilita la lectura y, fuera de `Non Refund`, se queda en la clase mayoritaria. No se reportan métricas: observar aciertos puntuales **no** equivale a validar el modelo (Unidad N° 6).
5. **Asociación, no causa.** Que `Non Refund` prediga bien cancelación en este CSV no autoriza a usarlo como palanca comercial. Cambiar la política de depósitos podría alterar el mix de clientes y el patrón dejaría de valer.

## 8. Cierre

Se recorrió el procedimiento mínimo de la Entrega 5 sobre `hotel_booking_TPI_grupo_J.csv` (comisión 10, Grupo J):

1. Predictores disponibles al registrar la reserva; se excluyeron variables con riesgo de fuga (`assigned_room_type`, `booking_changes`) y el identificador.
2. Partición train/test reproducible; imputación y dummies **solo** con información de entrenamiento.
3. Árbol de decisión de profundidad 3.
4. Predicciones en prueba, comparadas con el resultado real.
5. Interpretación, utilidad inicial y limitaciones.

La evaluación formal con métricas queda para la **Unidad N° 6**.

In [8]:
con.close()
print('Conexión DuckDB cerrada.')

Conexión DuckDB cerrada.
